# 01 - Data Exploration

Goal: inspect the IoT-23 CSV safely before preprocessing and graph construction.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "iot23_combined_new.csv"

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Data exists:", DATA_PATH.exists())

Python: c:\Users\Binh\OneDrive\Documents\CẦN NỘP\IDS_GAT_IOT23\.venv\Scripts\python.exe
Project root: c:\Users\Binh\OneDrive\Documents\CẦN NỘP\IDS_GAT_IOT23
Data path: c:\Users\Binh\OneDrive\Documents\CẦN NỘP\IDS_GAT_IOT23\data\iot23_combined_new.csv
Data exists: True


## Read a small sample

The full CSV is large, so first we load only a small sample.

In [2]:
df_sample = pd.read_csv(DATA_PATH, nrows=10_000)

# The Kaggle CSV contains an old saved index column.
df_sample = df_sample.drop(columns=["Unnamed: 0"], errors="ignore")

display(df_sample.head())
print("Shape:", df_sample.shape)

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label
0,1.536227e+09,CeqqKl3hyLQmO8LK98,192.168.100.111,17576.0,78.1.220.212,8081.0,tcp,-,0.000003,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
1,1.536227e+09,C2oHQWo1EFGH8D9x7,192.168.100.111,17576.0,152.84.7.111,8081.0,tcp,-,0.000002,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
2,1.536227e+09,CJLVjs4BByG04mczXc,192.168.100.111,17576.0,173.36.41.67,8081.0,tcp,-,0.000002,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
3,1.536227e+09,C0z4uS9AWHDH2s4S7,192.168.100.111,17576.0,87.13.21.104,8081.0,tcp,-,0.000002,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
4,1.536227e+09,CxbNVk3liFNUIlqSPi,192.168.100.111,17576.0,99.110.163.140,8081.0,tcp,-,0.000002,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan


Shape: (10000, 21)


In [3]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ts             10000 non-null  float64
 1   uid            10000 non-null  str    
 2   id.orig_h      10000 non-null  str    
 3   id.orig_p      10000 non-null  float64
 4   id.resp_h      10000 non-null  str    
 5   id.resp_p      10000 non-null  float64
 6   proto          10000 non-null  str    
 7   service        10000 non-null  str    
 8   duration       10000 non-null  float64
 9   orig_bytes     10000 non-null  int64  
 10  resp_bytes     10000 non-null  int64  
 11  conn_state     10000 non-null  str    
 12  local_orig     10000 non-null  str    
 13  local_resp     10000 non-null  str    
 14  missed_bytes   10000 non-null  float64
 15  history        10000 non-null  str    
 16  orig_pkts      10000 non-null  float64
 17  orig_ip_bytes  10000 non-null  float64
 18  resp_pkts      100

## Missing values and placeholder values

In [4]:
missing_report = (
    df_sample.replace("-", np.nan)
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_ratio")
)

missing_report

,missing_ratio
service,1.0
local_orig,1.0
local_resp,1.0
uid,0.0
ts,0.0
id.resp_h,0.0
id.orig_p,0.0
id.orig_h,0.0
proto,0.0
duration,0.0


## Label distribution on the sample

In [5]:
sample_label_counts = df_sample["label"].value_counts()
sample_label_ratio = df_sample["label"].value_counts(normalize=True)

pd.DataFrame({
    "count": sample_label_counts,
    "ratio": sample_label_ratio,
})

,count,ratio
label,,
PartOfAHorizontalPortScan,4996,0.4996
DDoS,2559,0.2559
Okiru,2441,0.2441
C&C-HeartBeat,2,0.0002
Benign,2,0.0002


## Label distribution on the full file

This reads only selected columns in chunks, so it is safer than loading the full CSV.

In [6]:
chunk_size = 500_000
total_rows = 0
label_counts = pd.Series(dtype="int64")
proto_counts = pd.Series(dtype="int64")
service_counts = pd.Series(dtype="int64")
conn_state_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["label", "proto", "service", "conn_state"],
    chunksize=chunk_size,
):
    total_rows += len(chunk)
    label_counts = label_counts.add(chunk["label"].value_counts(), fill_value=0)
    proto_counts = proto_counts.add(chunk["proto"].value_counts(), fill_value=0)
    service_counts = service_counts.add(chunk["service"].value_counts(), fill_value=0)
    conn_state_counts = conn_state_counts.add(chunk["conn_state"].value_counts(), fill_value=0)

label_counts = label_counts.astype(int).sort_values(ascending=False)

label_summary = pd.DataFrame({
    "count": label_counts,
    "ratio": label_counts / total_rows,
})

print("Total rows:", total_rows)
label_summary

Total rows: 6046623


,count,ratio
label,,
PartOfAHorizontalPortScan,3389036,5.604841e-01
Okiru,1313012,2.171480e-01
Benign,688812,1.139168e-01
DDoS,638506,1.055971e-01
C&C,15286,2.528023e-03
C&C-HeartBeat,1332,2.202883e-04
Attack,538,8.897528e-05
C&C-FileDownload,46,7.607552e-06
C&C-Torii,30,4.961447e-06


In [7]:
print("Protocol distribution")
display(proto_counts.astype(int).sort_values(ascending=False).to_frame("count"))

print("Top services")
display(service_counts.astype(int).sort_values(ascending=False).head(10).to_frame("count"))

print("Top connection states")
display(conn_state_counts.astype(int).sort_values(ascending=False).head(10).to_frame("count"))

Protocol distribution


,count
proto,
tcp,6026584
udp,18429
icmp,1610


Top services


,count
service,
-,6038628
dns,5723
irc,1655
http,344
dhcp,176
ssl,95
ssh,2


Top connection states


,count
conn_state,
S0,5514038
OTH,515339
SF,11653
REJ,2488
S3,2484
RSTR,334
SH,105
RSTO,100
RSTOS0,44


## First preprocessing decisions

These are initial decisions for the next notebook/script.

In [ ]:
target_col = "label"

drop_cols = [
    "uid",          # unique flow id, not useful as a model feature
    "local_orig",   # all '-' in this dataset
    "local_resp",   # all '-' in this dataset
]

ip_cols = ["id.orig_h", "id.resp_h"]
categorical_cols = ["proto", "service", "conn_state", "history"]

numeric_cols = [
    col
    for col in df_sample.columns
    if col not in drop_cols + ip_cols + categorical_cols + [target_col]
]

print("Target:", target_col)
print("Drop columns:", drop_cols)
print("IP columns:", ip_cols)
print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

Target: label
Drop columns: ['uid', 'local_orig', 'local_resp']
IP columns: ['id.orig_h', 'id.resp_h']
Categorical columns: ['proto', 'service', 'conn_state', 'history']
Numeric columns: ['ts', 'id.orig_p', 'id.resp_p', 'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes']


: 